# Notebook 2: Baseline FP-Growth và lọc hậu xử lý

Notebook này thực hiện kịch bản **Baseline (Post-processing)** cho bài thực nghiệm khai phá luật kết hợp định lượng.

Ý tưởng chính:

1. Dùng danh sách giao dịch đã tạo ở `data_preprocessing.ipynb`.
2. Chạy thuật toán FP-Growth nguyên bản, trong đó item được sắp xếp theo **support giảm dần** khi xây FP-Tree.
3. Sinh toàn bộ frequent itemsets theo từng ngưỡng `min_support`.
4. Sau khi mining xong mới lọc bỏ các itemset có `avg(price) > 20`.

Kết quả của notebook này sẽ được lưu lại để so sánh với thuật toán đề xuất trong các notebook tiếp theo.

## 1. Cấu hình môi trường và thư mục đầu ra

Notebook được thiết kế để chạy được trên cả local và Google Colab. Nếu chạy trên Colab, hãy chạy `data_preprocessing.ipynb` trước hoặc upload thư mục artifact đã sinh từ notebook 1. Có thể đổi thư mục artifact bằng biến môi trường `OUTPUT_DIR`, đổi danh sách support bằng `MIN_SUPPORT_RATIOS`, và đổi ngưỡng giá bằng `PRICE_THRESHOLD`.

In [1]:
import importlib.util
import json
import math
import os
import pickle
import subprocess
import sys
import time
from collections import Counter
from itertools import combinations
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    """Cài hoặc nạp thư viện cần thiết cho cả local và Colab."""
    pip_name = pip_name or import_name
    if importlib.util.find_spec(import_name) is None:
        bundled_packages = Path.home() / ".cache" / "codex-runtimes" / "codex-primary-runtime" / "dependencies" / "python"
        if bundled_packages.exists() and str(bundled_packages) not in sys.path:
            sys.path.insert(0, str(bundled_packages))

    if importlib.util.find_spec(import_name) is None:
        print(f"Thiếu thư viện '{import_name}'. Đang cài đặt gói '{pip_name}'...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])


ensure_package("pandas")

import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# =========================
# CẤU HÌNH LINH HOẠT
# =========================
# Có thể chỉnh bằng biến môi trường hoặc sửa trực tiếp tại đây khi chạy trên Colab/local.
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).resolve()
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR", PROJECT_ROOT / "outputs")).resolve()

ARTIFACT_FILES = {
    "transactions_pickle": os.environ.get("TRANSACTIONS_PICKLE", "transactions.pkl"),
    "item_price_pickle": os.environ.get("ITEM_PRICE_PICKLE", "item_price.pkl"),
    "preprocessing_summary_json": os.environ.get("PREPROCESSING_SUMMARY_JSON", "preprocessing_summary.json"),
    "baseline_results_pickle": os.environ.get("BASELINE_RESULTS_PICKLE", "baseline_results.pkl"),
    "baseline_metrics_csv": os.environ.get("BASELINE_METRICS_CSV", "baseline_metrics.csv"),
    "baseline_patterns_csv": os.environ.get("BASELINE_PATTERNS_CSV", "baseline_final_patterns.csv"),
    "baseline_summary_json": os.environ.get("BASELINE_SUMMARY_JSON", "baseline_summary.json"),
}


def parse_support_ratios(value, default_ratios):
    """Đọc danh sách support linh hoạt, ví dụ '5%,4%,3%' hoặc '0.05,0.04,0.03'."""
    if not value:
        return default_ratios
    ratios = []
    for token in value.split(","):
        token = token.strip()
        if not token:
            continue
        if token.endswith("%"):
            ratios.append(float(token[:-1]) / 100)
        else:
            number = float(token)
            ratios.append(number / 100 if number > 1 else number)
    return ratios


MIN_SUPPORT_RATIOS = parse_support_ratios(
    os.environ.get("MIN_SUPPORT_RATIOS", ""),
    default_ratios=[0.05, 0.04, 0.03, 0.02, 0.01],
)
PRICE_THRESHOLD = float(os.environ.get("PRICE_THRESHOLD", "20"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_rows", 3000)

print("Môi trường chạy:", "Google Colab" if IN_COLAB else "Local/Jupyter")
print("Thư mục làm việc:", PROJECT_ROOT)
print("Thư mục lưu kết quả:", OUTPUT_DIR)
print("Danh sách min_support:", [f"{ratio:.0%}" for ratio in MIN_SUPPORT_RATIOS])
print("Ngưỡng avg(price):", PRICE_THRESHOLD)

Môi trường chạy: Local/Jupyter
Thư mục làm việc: C:\Users\Acer\source\repos\homework\DataMining\Constraint-based_pattern_data_mining
Thư mục lưu kết quả: C:\Users\Acer\source\repos\homework\DataMining\Constraint-based_pattern_data_mining\outputs
Danh sách min_support: ['5%', '4%', '3%', '2%', '1%']
Ngưỡng avg(price): 20.0


## 2. Nạp dữ liệu tiền xử lý

Notebook dùng các artifact đã được tạo bởi `data_preprocessing.ipynb`:

- File giao dịch dạng pickle, mặc định `transactions.pkl`.
- File giá đại diện của từng item, mặc định `item_price.pkl`.
- File số liệu tổng hợp tiền xử lý, mặc định `preprocessing_summary.json`.

Các tên file mặc định có thể đổi trong cell cấu hình hoặc bằng biến môi trường.

In [3]:
def find_artifact(filename, output_dir=OUTPUT_DIR):
    """Tìm artifact theo tên file cấu hình trong OUTPUT_DIR, PROJECT_ROOT hoặc /content."""
    direct_candidates = [output_dir / filename, PROJECT_ROOT / filename]
    search_roots = [PROJECT_ROOT]
    if IN_COLAB:
        search_roots.append(Path("/content"))

    seen = set()
    for path in direct_candidates:
        resolved = path.resolve()
        if resolved not in seen:
            seen.add(resolved)
            if resolved.exists():
                return resolved

    for root in search_roots:
        if not root.exists():
            continue
        for path in root.glob(f"**/{filename}"):
            resolved = path.resolve()
            if resolved not in seen:
                seen.add(resolved)
                if resolved.exists():
                    return resolved

    return output_dir / filename


transactions_path = find_artifact(ARTIFACT_FILES["transactions_pickle"])
item_price_path = find_artifact(ARTIFACT_FILES["item_price_pickle"])
summary_path = find_artifact(ARTIFACT_FILES["preprocessing_summary_json"])

required_artifacts = [transactions_path, item_price_path, summary_path]
missing_artifacts = [str(path) for path in required_artifacts if not path.exists()]
if missing_artifacts:
    raise FileNotFoundError(
        "Thiếu artifact từ bước tiền xử lý. Hãy chạy data_preprocessing.ipynb trước. "
        f"Các file còn thiếu: {missing_artifacts}"
    )

with transactions_path.open("rb") as f:
    transactions = pickle.load(f)

with item_price_path.open("rb") as f:
    item_price = pickle.load(f)

with summary_path.open("r", encoding="utf-8") as f:
    preprocessing_summary = json.load(f)

load_summary_df = pd.DataFrame([
    {"Thông tin": "Số giao dịch", "Giá trị": len(transactions)},
    {"Thông tin": "Số item có giá đại diện", "Giá trị": len(item_price)},
    {"Thông tin": "Số dòng sau tiền xử lý", "Giá trị": preprocessing_summary.get("clean_rows")},
    {"Thông tin": "Số mặt hàng duy nhất", "Giá trị": preprocessing_summary.get("unique_items")},
])

print("Đã nạp dữ liệu tiền xử lý thành công.")
display(load_summary_df)

print("Ví dụ 5 giao dịch đầu:")
sample_transactions_df = pd.DataFrame({
    "Mã thứ tự giao dịch": list(range(1, 6)),
    "Số item": [len(t) for t in transactions[:5]],
    "Items": [" | ".join(t) for t in transactions[:5]],
})
display(sample_transactions_df)

Đã nạp dữ liệu tiền xử lý thành công.


,Thông tin,Giá trị
0,Số giao dịch,1000
1,Số item có giá đại diện,50
2,Số dòng sau tiền xử lý,1000
3,Số mặt hàng duy nhất,50


Ví dụ 5 giao dịch đầu:


,Mã thứ tự giao dịch,Số item,Items
0,1,3,ITEM_002 | ITEM_048 | ITEM_018
1,2,5,ITEM_015 | ITEM_009 | ITEM_048 | ITEM_007 | ITEM_044
2,3,10,ITEM_006 | ITEM_038 | ITEM_028 | ITEM_003 | ITEM_002 | ITEM_050 | ITEM_014 | ITEM_015 | ITEM_033 | ITEM_039
3,4,2,ITEM_036 | ITEM_013
4,5,10,ITEM_027 | ITEM_015 | ITEM_029 | ITEM_038 | ITEM_018 | ITEM_001 | ITEM_011 | ITEM_028 | ITEM_022 | ITEM_046


## 3. Cấu hình thực nghiệm Baseline

Các ngưỡng `min_support` được đọc từ cấu hình `MIN_SUPPORT_RATIOS`, mặc định là `5%`, `4%`, `3%`, `2%`, `1%`. Với mỗi ngưỡng, notebook đổi sang số lượng giao dịch tối thiểu bằng công thức:

`min_support_count = ceil(min_support_ratio * số_giao_dịch)`

Điều kiện định lượng dùng ở bước hậu xử lý là `avg(price) <= PRICE_THRESHOLD`, mặc định `20`.

In [4]:
N_TRANSACTIONS = len(transactions)

support_config_df = pd.DataFrame([
    {
        "Min Support": f"{ratio:.0%}",
        "Tỷ lệ": ratio,
        "Support count tối thiểu": math.ceil(ratio * N_TRANSACTIONS),
    }
    for ratio in MIN_SUPPORT_RATIOS
])

print("Cấu hình thực nghiệm:")
display(support_config_df)
print(f"Ngưỡng lọc hậu xử lý: avg(price) <= {PRICE_THRESHOLD}")

Cấu hình thực nghiệm:


,Min Support,Tỷ lệ,Support count tối thiểu
0,5%,0.05,50
1,4%,0.04,40
2,3%,0.03,30
3,2%,0.02,20
4,1%,0.01,10


Ngưỡng lọc hậu xử lý: avg(price) <= 20.0


## 4. Cài đặt FP-Growth Baseline

Phần này cài đặt FP-Growth theo đúng tinh thần baseline:

- Tính support của item đơn.
- Giữ các item đạt `min_support`.
- Sắp xếp item trong mỗi giao dịch theo **support giảm dần** khi xây FP-Tree.
- Sinh toàn bộ frequent itemsets bằng pattern growth.
- Chỉ sau khi sinh xong mới lọc theo điều kiện `avg(price) <= 20`.

Cài đặt có thêm tối ưu single-path để tăng tốc khi cây điều kiện chỉ còn một đường đi.

In [5]:
class FPNode:
    """Một node trong FP-Tree."""

    __slots__ = ("item", "count", "parent", "children", "link")

    def __init__(self, item, count=0, parent=None):
        self.item = item
        self.count = count
        self.parent = parent
        self.children = {}
        self.link = None


class FPTree:
    """FP-Tree kèm header table để truy xuất nhanh các node cùng item."""

    __slots__ = ("root", "headers", "supports")

    def __init__(self):
        self.root = FPNode(item=None, count=0, parent=None)
        self.headers = {}
        self.supports = {}

    def add_transaction(self, ordered_items, count=1):
        current = self.root
        for item in ordered_items:
            child = current.children.get(item)
            if child is None:
                child = FPNode(item=item, count=0, parent=current)
                current.children[item] = child
                child.link = self.headers.get(item)
                self.headers[item] = child
            child.count += count
            current = child

    def is_single_path(self):
        current = self.root
        while True:
            if len(current.children) > 1:
                return False
            if not current.children:
                return True
            current = next(iter(current.children.values()))

    def single_path_nodes(self):
        nodes = []
        current = self.root
        while current.children:
            current = next(iter(current.children.values()))
            nodes.append(current)
        return nodes


def build_fp_tree(weighted_transactions, min_support_count):
    """Xây FP-Tree theo thứ tự support giảm dần."""
    item_counts = Counter()
    for items, weight in weighted_transactions:
        for item in set(items):
            item_counts[item] += weight

    frequent_supports = {
        item: count
        for item, count in item_counts.items()
        if count >= min_support_count
    }
    if not frequent_supports:
        return None

    item_rank = {
        item: rank
        for rank, (item, _) in enumerate(sorted(frequent_supports.items(), key=lambda pair: (-pair[1], pair[0])))
    }

    tree = FPTree()
    tree.supports = frequent_supports

    for items, weight in weighted_transactions:
        filtered_items = [item for item in set(items) if item in frequent_supports]
        if filtered_items:
            filtered_items.sort(key=lambda item: item_rank[item])
            tree.add_transaction(filtered_items, count=weight)

    return tree


def conditional_pattern_base(tree, item):
    """Lấy conditional pattern base của một item từ FP-Tree."""
    base = []
    node = tree.headers.get(item)

    while node is not None:
        path = []
        parent = node.parent
        while parent is not None and parent.item is not None:
            path.append(parent.item)
            parent = parent.parent
        if path:
            path.reverse()
            base.append((path, node.count))
        node = node.link

    return base


def mine_fp_tree(tree, min_support_count, suffix=()):
    """Sinh toàn bộ frequent itemsets từ FP-Tree."""
    patterns = {}
    if tree is None:
        return patterns

    if tree.is_single_path():
        nodes = tree.single_path_nodes()
        path_items = [node.item for node in nodes]
        path_counts = [node.count for node in nodes]

        for size in range(1, len(path_items) + 1):
            for indices in combinations(range(len(path_items)), size):
                itemset = tuple(sorted(tuple(path_items[i] for i in indices) + suffix))
                support_count = min(path_counts[i] for i in indices)
                patterns[itemset] = support_count
        return patterns

    # FP-Growth thường duyệt từ item ít phổ biến hơn trong header table để tạo cây điều kiện.
    for item, support_count in sorted(tree.supports.items(), key=lambda pair: (pair[1], pair[0])):
        new_suffix = tuple(sorted(suffix + (item,)))
        patterns[new_suffix] = support_count

        conditional_base = conditional_pattern_base(tree, item)
        conditional_tree = build_fp_tree(conditional_base, min_support_count)
        if conditional_tree is not None:
            patterns.update(mine_fp_tree(conditional_tree, min_support_count, new_suffix))

    return patterns


def average_itemset_price(itemset, price_lookup):
    """Tính avg(price) của một itemset theo bảng giá đại diện."""
    return sum(float(price_lookup[item]) for item in itemset) / len(itemset)


def run_baseline_fp_growth(transactions, item_price, min_support_ratio, price_threshold=None):
    """Chạy FP-Growth baseline rồi lọc hậu xử lý theo avg(price)."""
    if price_threshold is None:
        price_threshold = PRICE_THRESHOLD
    min_support_count = math.ceil(min_support_ratio * len(transactions))
    weighted_transactions = [(transaction, 1) for transaction in transactions]

    start_time = time.perf_counter()
    tree = build_fp_tree(weighted_transactions, min_support_count)
    all_patterns = mine_fp_tree(tree, min_support_count)
    final_patterns = {
        itemset: support_count
        for itemset, support_count in all_patterns.items()
        if average_itemset_price(itemset, item_price) <= price_threshold
    }
    runtime_seconds = time.perf_counter() - start_time

    return {
        "min_support_ratio": min_support_ratio,
        "min_support_label": f"{min_support_ratio:.0%}",
        "min_support_count": min_support_count,
        "runtime_seconds": runtime_seconds,
        "all_patterns": all_patterns,
        "final_patterns": final_patterns,
        "num_all_patterns": len(all_patterns),
        "num_final_patterns": len(final_patterns),
        "num_filtered_patterns": len(all_patterns) - len(final_patterns),
    }


print("Đã định nghĩa xong cấu trúc FP-Tree và hàm chạy Baseline FP-Growth.")

Đã định nghĩa xong cấu trúc FP-Tree và hàm chạy Baseline FP-Growth.
